In [ ]:
import os
import numpy as np
import pandas as pd
import mysql.connector
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Chargement des variables d'environnement depuis le fichier .env du dossier parent
load_dotenv(dotenv_path="../.env")

In [ ]:
# Connexion à la base de données Cloud Aiven MySQL
connection = mysql.connector.connect(
    host=os.getenv("AIVEN_HOST"),
    port=int(os.getenv("AIVEN_PORT", 14198)),
    user=os.getenv("AIVEN_USER"),
    password=os.getenv("AIVEN_PASSWORD"),
    database="defaultdb"
)

In [ ]:
# 1. Extraction d'un échantillon propre
# Note: On utilise la station 3012 (Saxe-Gambetta) ou 1034 qui contiennent des milliers d'enregistrements.
query = """
SELECT date_enregistrement AS date_enregistrement, velos 
FROM stations_historique 
WHERE id = 3012 
ORDER BY date_enregistrement ASC
"""

In [ ]:
# 2. Chargement dans un DataFrame
df = pd.read_sql(query, connection)
connection.close()

In [ ]:
# Vérification rapide du chargement
print("Colonnes chargées :", list(df.columns))
print("Nombre de lignes brutes :", len(df))
if len(df) > 0:
    print(df.head())

In [ ]:
# 3. Préparation des données (Index temporel + rééchantillonnage dynamique)
if 'date_enregistrement' in df.columns:
    df['date_enregistrement'] = pd.to_datetime(df['date_enregistrement'])
    df.set_index('date_enregistrement', inplace=True)

# Calcul de la durée totale des données chargées en heures
time_span_hours = (df.index.max() - df.index.min()).total_seconds() / 3600

# Choix dynamique de la fréquence de rééchantillonnage pour conserver assez de points (min 48)
if time_span_hours < 8:
    freq = '1T'  # Toutes les minutes
    print(f"Durée des données courte ({time_span_hours:.1f}h). Rééchantillonnage par minute ('1T')")
elif time_span_hours < 36:
    freq = '10T' # Toutes les 10 minutes
    print(f"Durée des données moyenne ({time_span_hours:.1f}h). Rééchantillonnage par 10 minutes ('10T')")
elif time_span_hours < 120:
    freq = '30T' # Toutes les 30 minutes
    print(f"Durée des données ({time_span_hours:.1f}h). Rééchantillonnage par 30 minutes ('30T')")
else:
    freq = 'h'   # Toutes les heures
    print(f"Durée des données longue ({time_span_hours:.1f}h). Rééchantillonnage par heure ('h')")

df = df.resample(freq).mean().ffill()
print(f"Nombre de points rééchantillonnés : {len(df)}")

In [ ]:
# 4. Mise à l'échelle et formatage des séquences pour le LSTM (Mise en forme 3D)
if len(df) >= 30:
    # Les réseaux LSTM sont très sensibles aux échelles des données, on les normalise entre 0 et 1
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(df[['velos']])
    
    # Définition de la taille de la fenêtre historique (look_back)
    # On utilise les 12 points précédents pour prédire le suivant
    look_back = 12
    
    X, y = [], []
    for i in range(len(scaled_data) - look_back):
        X.append(scaled_data[i:(i + look_back), 0])
        y.append(scaled_data[i + look_back, 0])
        
    X, y = np.array(X), np.array(y)
    
    # Reshape pour correspondre au format requis par Keras LSTM : [samples, time_steps, features]
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))
    
    print(f"Dimensions de la matrice X (LSTM 3D) : {X.shape}")
    print(f"Dimensions du vecteur cible y : {y.shape}")
else:
    print("Pas assez de données pour structurer le modèle LSTM.")

In [ ]:
# 5. Split Train/Test et Entraînement du réseau LSTM (Deep Learning)
if 'X' in locals() and len(X) >= 20:
    # Split temporel (80% entraînement, 20% test)
    train_size = int(len(X) * 0.8)
    X_train, X_test = X[:train_size], X[train_size:]
    y_train, y_test = y[:train_size], y[train_size:]
    
    print(f"Entraînement sur {len(X_train)} séquences, validation sur {len(X_test)} séquences.")
    
    # Construction du modèle Keras LSTM
    model_lstm = Sequential()
    # Couche LSTM
    model_lstm.add(LSTM(50, return_sequences=False, input_shape=(look_back, 1)))
    # Couche de sortie Dense
    model_lstm.add(Dense(1))
    
    model_lstm.compile(optimizer='adam', loss='mean_squared_error')
    
    # Entraînement du modèle
    print("Début de l'entraînement du réseau de neurones (LSTM)... (20 époques)")
    history = model_lstm.fit(
        X_train, y_train, 
        epochs=20, 
        batch_size=16, 
        validation_split=0.1, 
        verbose=1
    )
    
    # Graphique de la convergence (Training Loss vs Validation Loss)
    plt.figure(figsize=(10, 5))
    plt.plot(history.history['loss'], label='Perte Entraînement (Loss)')
    plt.plot(history.history['val_loss'], label='Perte Validation (Val Loss)')
    plt.title('Courbe d\'apprentissage du modèle LSTM')
    plt.xlabel('Époques')
    plt.ylabel('Erreur Quadratique Moyenne (MSE)')
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("Pas assez de données pour entraîner le modèle LSTM.")

In [ ]:
# 6. Prédictions et Évaluation du réseau de neurones
if 'model_lstm' in locals() and 'X_test' in locals():
    # Prédictions sur le jeu de test (données normalisées)
    predictions_scaled = model_lstm.predict(X_test)
    
    # Transformation inverse pour revenir à l'échelle réelle (nombre de vélos)
    predictions = scaler.inverse_transform(predictions_scaled)
    y_test_real = scaler.inverse_transform(y_test.reshape(-1, 1))
    y_train_real = scaler.inverse_transform(y_train.reshape(-1, 1))
    
    # Calcul des erreurs
    mae = mean_absolute_error(y_test_real, predictions)
    rmse = np.sqrt(mean_squared_error(y_test_real, predictions))
    
    print(f"\n--- Métriques de validation (Réseau de neurones LSTM) ---")
    print(f"Erreur Moyenne Absolue (MAE) : {mae:.2f} vélos")
    print(f"Erreur Quadratique Moyenne (RMSE) : {rmse:.2f} vélos")
    
    # Dates correspondantes à la période de test
    test_dates = df.index[look_back + train_size:]
    train_dates = df.index[look_back:look_back + train_size]
    
    # Graphique final de comparaison
    plt.figure(figsize=(12, 6))
    plt.plot(train_dates, y_train_real, label='Entraînement', color='blue')
    plt.plot(test_dates, y_test_real, label='Valeurs Réelles', color='green')
    plt.plot(test_dates, predictions, label='Prédictions LSTM', color='red', linestyle='--')
    plt.title("Modèle Deep Learning : Réseau de neurones LSTM")
    plt.xlabel("Date / Heure")
    plt.ylabel("Nombre de vélos")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("Prédiction impossible car le modèle n'a pas été entraîné.")